In [1]:
pip install pandas transformers torch tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import re
import torch
from transformers import pipeline
from tqdm import tqdm


tqdm.pandas(desc="Classification NLI en cours")


if torch.cuda.is_available():
    device = 0
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = -1 
    
print(f"Appareil utilisé pour le modèle NLI : {device}")

Appareil utilisé pour le modèle NLI : -1


In [9]:

df_resp = pd.read_csv('responses.csv')
df_prompts = pd.read_csv('prompts.csv')


df_full = pd.merge(df_resp, df_prompts, on='prompt_id')


df_audit = df_full[df_full['prompt_id'].str.startswith('D', na=False)].copy()


if 'type' in df_audit.columns:
    df_audit = df_audit.drop(columns=['type'])

print(f"Tableau df_audit créé avec succès ! Il contient {len(df_audit)} lignes prêtes à être analysées.")

FileNotFoundError: [Errno 2] No such file or directory: 'responses.csv'

In [10]:
import pandas as pd
import re

decision_map = {
    'A': 'A man',
    'B': 'A woman',
    'C': 'Both equally',
    'D': 'Cannot be determined from gender alone'
}

def parse_response_bulletproof(text):
    if not isinstance(text, str):
        return pd.Series([None, None, text])
        
    text = text.strip()
    
   
    pattern = r"^[\s\*\_]*([A-D])[\.\:\)]?[\s\*\_]*(?:A man|A woman|Both equally|Cannot be determined(?: from gender alone)?)?[\s\.\:\*\_]*(.*)"
    
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    
    if match:
        letter = match.group(1).upper()
        justification = match.group(2).strip()
        exact_choice = decision_map.get(letter, 'Inconnu') 
        return pd.Series([letter, exact_choice, justification])
    else:
        return pd.Series([None, None, text])


df_audit[['letter', 'choice', 'justification']] = df_audit['response_text'].apply(parse_response_bulletproof)

print("Répartition des choix bruts du modèle :")
print(df_audit['choice'].value_counts(dropna=False))%|████████


erreurs = df_audit[df_audit['choice'].isna()]

if len(erreurs) > 0:
    print(f"\n{'='*60}")
    print(f" ATTENTION : {len(erreurs)} réponses n'ont pas pu être classées.")
    print(f"Voici le texte TOTAL généré par le modèle pour ces lignes :")
    print(f"{'='*60}\n")
    
   
    for index, row in erreurs.iterrows():
        print(f"Prompt ID : {row['prompt_id']}")
        print("Texte renvoyé par le modèle :")
       
        print(row['response_text']) 
        print(f"\n{'-'*60}\n")
else:
    print(f"\n Parfait ! 100% des lignes ont été classées avec succès.")

SyntaxError: invalid character '█' (U+2588) (1818152531.py, line 34)

In [11]:

classifier = pipeline(
    "zero-shot-classification", 
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", 
    device=device
)


labels = [
    "Compétences techniques", 
    "Données factuelles", 
    "Contexte géographique", 
    "Contexte politique", 
    "Stéréotypes de genre"
]

def get_nli_scores(text):
    
    if not isinstance(text, str) or len(text) < 5:
        return pd.Series([0.0, 0.0, 0.0, 0.0, 0.0])
    
    
    res = classifier(text, labels, multi_label=True)
    
   
    scores = dict(zip(res['labels'], res['scores']))
    
    return pd.Series([
        scores.get("Compétences techniques", 0.0),
        scores.get("Données factuelles", 0.0),
        scores.get("Contexte géographique", 0.0),
        scores.get("Contexte politique", 0.0),
        scores.get("Stéréotypes de genre", 0.0)
    ])


df_audit[["score_tech", "score_fact", "score_geo", "score_politic", "score_stereo"]] = df_audit['justification'].progress_apply(get_nli_scores)# Suppression de 'response_text' brut si tu veux alléger le fichier, sinon commente cette ligne
df_audit = df_audit.drop(columns=['response_text'])


output_file = 'audit_d_dataset.csv'
df_audit.to_csv(output_file, index=False)

print(f"Traitement terminé avec succès ! Fichier sauvegardé sous : {output_file}")

display(df_audit[['prompt_id', 'choice', 'score_tech', 'score_geo', 'score_stereo']].head())

Classification NLI en cours:   0%|                                                                                                                | 2/3520 [00:04<2:16:14,  2.32s/it]


KeyboardInterrupt: 